# Stage B — Transdiagnostic Patient Embedding, Clustering & Validation

**Comprehensive notebook** covering the full Stage B pipeline:

1. Data loading & inspection
2. Train/test split with diagnostics
3. Embedding method comparison (16 methods)
4. Clustering comparison (10 methods × k sweep)
5. Validation (internal, external, info-theoretic, permutation)
6. Stability analysis (bootstrap, perturbation, LOCO)
7. Interpretability (feature importance, block ablation)
8. Clinical interpretation (enrichment, DSM comparison)
9. Final summary

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import logging
logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from pathlib import Path
DATA_DIR = Path('../data')
OUTPUT_DIR = Path('../output/stage_b')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading & Inspection

In [2]:
from face_stratification import build_harmonized_dataset, load_feature_schema

schema = load_feature_schema()
csv_paths = {
    'bp': DATA_DIR / 'BP.csv',
    'sz': DATA_DIR / 'SZ.csv',
    'dr': DATA_DIR / 'DR.csv',
    'asp': DATA_DIR / 'ASP.csv',
}

dataset = build_harmonized_dataset(csv_paths, schema=schema)
print(f'Patients: {dataset.n_patients}, Features: {dataset.n_features}')
print(f'Cohorts:\n{dataset.cohort_counts()}')

face_stratification.harmonization.harmonizer | Loading cohort bp from ../data/BP.csv
face_stratification.harmonization.harmonizer | Cohort bp: 6252 rows
face_stratification.harmonization.harmonizer | Loading cohort sz from ../data/SZ.csv
face_stratification.harmonization.harmonizer | Cohort sz: 2209 rows
face_stratification.harmonization.harmonizer | Loading cohort dr from ../data/DR.csv
face_stratification.harmonization.harmonizer | Cohort dr: 552 rows
face_stratification.harmonization.harmonizer | Loading cohort asp from ../data/ASP.csv
face_stratification.harmonization.harmonizer | Cohort asp: 2001 rows


Patients: 11014, Features: 184
Cohorts:
cohort
asp    2001
bp     6252
dr      552
sz     2209
Name: count, dtype: int64


In [3]:
# Feature coverage heatmap
coverage = dataset.feature_availability()
print(f'Features with >50% coverage in all cohorts:')
cov_cols = [c for c in coverage.columns if c.startswith('coverage_') and c != 'coverage_total']
high_cov = coverage[(coverage[cov_cols] > 0.5).all(axis=1)]
print(f'  {len(high_cov)} / {len(coverage)} features')
high_cov[['coverage_total'] + cov_cols].head(15)

Features with >50% coverage in all cohorts:
  9 / 184 features


,coverage_total,coverage_asp,coverage_bp,coverage_dr,coverage_sz
feature_id,,,,,
demo_age_years,1.0,1.0,1.0,1.0,1.0
demo_sex_male,1.0,1.0,1.0,1.0,1.0
sub_tobacco_current,1.0,1.0,1.0,1.0,1.0
sub_alcohol_current,1.0,1.0,1.0,1.0,1.0
sub_cannabis_current,1.0,1.0,1.0,1.0,1.0
sub_use_disorder,1.0,1.0,1.0,1.0,1.0
cm_n_somatic,1.0,1.0,1.0,1.0,1.0
cm_n_psychiatric,1.0,1.0,1.0,1.0,1.0
tx_polypharmacy_index,1.0,1.0,1.0,1.0,1.0


## 2. Train/Test Split

In [4]:
from face_stratification.evaluation.split import create_stratified_split

split = create_stratified_split(dataset, test_fraction=0.2, seed=42)
print(f'Train: {split.n_train}, Test: {split.n_test}')
print(f'\nTrain cohort counts: {split.metadata["train_cohort_counts"]}')
print(f'Test cohort counts:  {split.metadata["test_cohort_counts"]}')

face_stratification.evaluation.split | Created stratified split: 8811 train / 2203 test (seed=42)
face_stratification.evaluation.split |   asp: 1600 train / 401 test (20.0%)
face_stratification.evaluation.split |   bp: 5002 train / 1250 test (20.0%)
face_stratification.evaluation.split |   dr: 442 train / 110 test (19.9%)
face_stratification.evaluation.split |   sz: 1767 train / 442 test (20.0%)


Train: 8811, Test: 2203

Train cohort counts: {'bp': 5002, 'sz': 1767, 'asp': 1600, 'dr': 442}
Test cohort counts:  {'bp': 1250, 'sz': 442, 'asp': 401, 'dr': 110}


In [5]:
# Verify stratification preserves proportions
train_ds = split.train_dataset(dataset)
test_ds = split.test_dataset(dataset)

train_props = train_ds.metadata['cohort'].value_counts(normalize=True).sort_index()
test_props = test_ds.metadata['cohort'].value_counts(normalize=True).sort_index()
pd.DataFrame({'Train %': (train_props * 100).round(1), 'Test %': (test_props * 100).round(1)})

,Train %,Test %
cohort,,
asp,18.2,18.2
bp,56.8,56.7
dr,5.0,5.0
sz,20.1,20.1


## 3. Normalization & Graph (train only)

In [6]:
from face_stratification.harmonization.normalization import fit_normalization, transform_normalization
from face_stratification.harmonization.harmonizer import HarmonizedDataset

# Fit normalization on TRAIN ONLY
norm_stats = fit_normalization(train_ds.X, dataset.schema)
train_Xn = transform_normalization(train_ds.X, norm_stats)
test_Xn = transform_normalization(test_ds.X, norm_stats)

train_ds_norm = HarmonizedDataset(
    X=train_Xn, metadata=train_ds.metadata,
    feature_metadata=train_ds.feature_metadata, schema=train_ds.schema,
)

print(f'Normalized train: {train_Xn.shape}')
print(f'Normalized test: {test_Xn.shape}')

Normalized train: (8811, 184)
Normalized test: (2203, 184)


In [7]:
# Build graph on TRAIN ONLY
from face_stratification.graph.patient_similarity import build_multiplex_graph

graph, block_graphs, td_result = build_multiplex_graph(train_Xn, dataset.schema, k=10, metadata=train_ds.metadata)
print(f'Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')
print(f'Block graphs built: {len(block_graphs)}')

face_stratification.graph.transdiagnostic | Transdiagnostic selection: 9 / 184 features passed (4.9%)
face_stratification.graph.patient_similarity | Block weight normalization: median total weight=16470.20, 22 blocks rescaled (range 176.5176–87021.5810 → all ≈16470.20)


Graph: 8811 nodes, 915965 edges
Block graphs built: 21


## 4. Embedding Method Comparison

Fit a subset of methods for this notebook. The full 16-method comparison is run via `StageBPipeline`.

In [8]:
import time
from face_stratification.models.baselines import TransdiagnosticPCA
from face_stratification.models.kernel_methods import KernelPCAEmbedding
from face_stratification.models.spectral import TransdiagnosticSpectral, MultiplexSpectral
from face_stratification.models.composite import ConcatenatedEmbedding

methods = {
    'PCA': TransdiagnosticPCA(n_components=8, l2_normalize=True),
    'KernelPCA': KernelPCAEmbedding(n_components=16, kernel='rbf'),
    'Spectral (TD)': TransdiagnosticSpectral(n_components=16, l2_normalize=True),
    'Spectral (Multiplex)': MultiplexSpectral(n_components=32, l2_normalize=True),
    'Composite': ConcatenatedEmbedding.build_default(),
}

embeddings = {}
for name, model in methods.items():
    t0 = time.time()
    needs_graph = name not in ('PCA', 'KernelPCA')
    emb = model.fit_transform(train_ds_norm, graph=graph if needs_graph else None)
    elapsed = time.time() - t0
    embeddings[name] = emb
    print(f'{name}: {emb.n_patients} x {emb.dim} in {elapsed:.1f}s')

face_stratification.graph.transdiagnostic | Transdiagnostic selection: 9 / 184 features passed (4.9%)


PCA: 8811 x 8 in 0.0s


face_stratification.models.kernel_methods | KernelPCA: 8811 patients → 16 dims (kernel=rbf)


KernelPCA: 8811 x 16 in 15.9s


face_stratification.models.spectral | eigsh shift-invert failed (Factor is exactly singular); retrying with which='SM'
face_stratification.models.spectral | eigsh which='SM' failed (ARPACK error -1: No convergence (85601 iterations, 0/17 eigenvectors converged)); falling back to dense eigh


Spectral (TD): 8811 x 16 in 403.1s


face_stratification.models.composite | ConcatenatedEmbedding: fitting sub-view 'transdiagnostic_pca'
face_stratification.graph.transdiagnostic | Transdiagnostic selection: 9 / 184 features passed (4.9%)
face_stratification.models.composite | ConcatenatedEmbedding: fitting sub-view 'transdiagnostic_spectral'


Spectral (Multiplex): 8811 x 32 in 46.6s


face_stratification.models.spectral | eigsh shift-invert failed (Factor is exactly singular); retrying with which='SM'
face_stratification.models.spectral | eigsh which='SM' failed (ARPACK error -1: No convergence (85601 iterations, 0/17 eigenvectors converged)); falling back to dense eigh
face_stratification.models.composite | ConcatenatedEmbedding: fitting sub-view 'multiplex_spectral'


Composite: 8811 x 56 in 463.3s


## 5. Clustering Comparison

In [9]:
from face_stratification.clustering.algorithms import kmeans_sweep, run_kmeans

cohort_labels = train_ds.metadata['cohort'].values

sweep_results = {}
for name, emb in embeddings.items():
    sweep = kmeans_sweep(emb.values, k_values=range(3, 13), reference_labels=cohort_labels)
    sweep_results[name] = sweep
    best_k = int(sweep.loc[sweep['silhouette'].idxmax(), 'k'])
    best_sil = sweep['silhouette'].max()
    print(f'{name}: best k={best_k}, silhouette={best_sil:.3f}')

PCA: best k=6, silhouette=0.423
KernelPCA: best k=9, silhouette=0.441
Spectral (TD): best k=12, silhouette=0.156
Spectral (Multiplex): best k=3, silhouette=0.489
Composite: best k=3, silhouette=0.211


### 5.1 Silhouette vs Davies–Bouldin — multi-metric k selection

**Silhouette alone is unreliable** for this dataset. It prefers solutions that maximize intra-cluster cohesion,
which on a cohort-biased graph (17 blocks, most cohort-specific) means it rewards solutions that simply
rediscover DSM labels. Davies–Bouldin (lower = better) often disagrees and can flag the opposite failure mode.

We add a composite score: `silhouette + 1/(1+DB) + transdiagnostic_entropy_ratio - 0.5 * cramers_v`.
The transdiagnostic term pushes away from DSM-aligned solutions; the Cramér's V penalty punishes over-alignment.

In [10]:
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score,
)

def _cramers_v(labels, ref):
    from scipy.stats import chi2_contingency
    ct = pd.crosstab(pd.Series(labels), pd.Series(ref))
    n = ct.sum().sum()
    chi2, _, _, _ = chi2_contingency(ct)
    k = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 and n > 0 else 0.0

def _per_cluster_entropy_ratio(labels, cohort_labels):
    """Mean per-cluster cohort entropy / log2(n_cohorts). 1.0 = perfectly transdiagnostic."""
    import math
    n_cohorts = len(np.unique(cohort_labels))
    max_h = math.log2(max(n_cohorts, 2))
    entropies = []
    for c in np.unique(labels):
        coh_in_c = cohort_labels[labels == c]
        _, counts = np.unique(coh_in_c, return_counts=True)
        p = counts / counts.sum()
        entropies.append(float(-(p * np.log2(p + 1e-15)).sum()))
    return float(np.mean(entropies) / max_h) if max_h > 0 else 0.0

def multi_metric_sweep(emb_values, cohort_labels, k_values=range(3, 13), sil_sample=5000):
    arr = emb_values.to_numpy(dtype=np.float64)
    rows = []
    rng = np.random.default_rng(0)
    sample_idx = rng.choice(len(arr), min(sil_sample, len(arr)), replace=False)
    for k in k_values:
        km = KMeans(n_clusters=k, random_state=0, n_init=10)
        labs = km.fit_predict(arr)
        sil = silhouette_score(arr[sample_idx], labs[sample_idx], metric='cosine')
        db = davies_bouldin_score(arr, labs)
        ch = calinski_harabasz_score(arr, labs)
        ari = adjusted_rand_score(cohort_labels, labs)
        nmi = normalized_mutual_info_score(cohort_labels, labs)
        cv = _cramers_v(labs, cohort_labels)
        td_ratio = _per_cluster_entropy_ratio(labs, cohort_labels)
        composite = sil + 1.0 / (1.0 + db) + td_ratio - 0.5 * cv
        rows.append(dict(
            k=k, silhouette=sil, davies_bouldin=db, calinski_harabasz=ch,
            ari=ari, nmi=nmi, cramers_v=cv, td_ratio=td_ratio, composite=composite,
        ))
    return pd.DataFrame(rows)

multi_metric_results = {}
for name, emb in embeddings.items():
    multi_metric_results[name] = multi_metric_sweep(emb.values, cohort_labels, k_values=range(3, 13))

# Summary: best k per metric per method
rows = []
for name, df in multi_metric_results.items():
    rows.append({
        'Method': name,
        'k (silhouette)': int(df.loc[df['silhouette'].idxmax(), 'k']),
        'k (DB)': int(df.loc[df['davies_bouldin'].idxmin(), 'k']),  # lower is better
        'k (CH)': int(df.loc[df['calinski_harabasz'].idxmax(), 'k']),
        'k (composite)': int(df.loc[df['composite'].idxmax(), 'k']),
        'best sil': df['silhouette'].max(),
        'best DB': df['davies_bouldin'].min(),
        'best composite': df['composite'].max(),
    })
pd.DataFrame(rows).set_index('Method').style.format('{:.3f}',
    subset=['best sil', 'best DB', 'best composite'])


,k (silhouette),k (DB),k (CH),k (composite),best sil,best DB,best composite
Method,,,,,,,
PCA,6,5,5,5,0.423,1.190,1.319
KernelPCA,9,9,3,3,0.441,1.216,1.097
Spectral (TD),12,12,3,10,0.156,2.223,1.162
Spectral (Multiplex),3,4,3,4,0.489,1.581,0.827
Composite,3,9,3,6,0.211,2.415,0.543


In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DARK = dict(template='plotly_dark', paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27')
COLORS = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA']

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Silhouette (higher is better)', 'Davies–Bouldin (LOWER is better)', 'Composite score (higher is better)'),
    horizontal_spacing=0.08,
)

for i, (name, df) in enumerate(multi_metric_results.items()):
    color = COLORS[i % len(COLORS)]
    fig.add_trace(go.Scatter(x=df['k'], y=df['silhouette'], name=name,
                              line=dict(color=color), mode='lines+markers',
                              legendgroup=name), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['k'], y=df['davies_bouldin'], name=name,
                              line=dict(color=color), mode='lines+markers',
                              legendgroup=name, showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=df['k'], y=df['composite'], name=name,
                              line=dict(color=color), mode='lines+markers',
                              legendgroup=name, showlegend=False), row=1, col=3)

fig.update_layout(**DARK, height=450, width=1400, title_text='k Selection — Silhouette vs Davies–Bouldin vs Composite')
fig.update_xaxes(title_text='k')
fig.update_yaxes(title_text='Silhouette', row=1, col=1)
fig.update_yaxes(title_text='DB', row=1, col=2)
fig.update_yaxes(title_text='Composite', row=1, col=3)
fig.write_image(str(OUTPUT_DIR / 'sil_vs_db_vs_composite.png'), width=1400, height=450, scale=3)
fig.show()


choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpf4l25jg5.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpx8au20a2.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpx8au20a2
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpf4l25jg5/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navigates done, putting them all in queue.
kaleido.kaleido | Getting tab from queue (has 1)
kaleido.kaleido | Got CF12
kaleido._kaleido_tab | Processing k_Selection__Silhouette_vs_DaviesBouldin_vs_Composite.png
kaleido._kaleido_tab | Sendin

In [12]:
# Stability curves
from face_stratification.visualization.stage_b_plots import plot_stability_curves

fig = plot_stability_curves(sweep_results, metric='silhouette',
                           output_path=OUTPUT_DIR / 'silhouette_vs_k.png')
fig.show()

choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpu4vyo866.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpafqtpy9c.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpafqtpy9c
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpu4vyo866/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navigates done, putting them all in queue.
kaleido.kaleido | Getting tab from queue (has 1)
kaleido.kaleido | Got B546
kaleido._kaleido_tab | Processing Stability_Curves__Silhouette_vs_k.png
kaleido._kaleido_tab | Sending big command for St

## 6. Validation — Best Configuration

In [13]:
# Pick the best method+k and run full validation
best_name = 'Composite'
best_emb = embeddings[best_name]
best_k = int(sweep_results[best_name].loc[
    sweep_results[best_name]['silhouette'].idxmax(), 'k'
])

assignment = run_kmeans(best_emb.values, n_clusters=best_k, reference_labels=cohort_labels)
print(f'Best: {best_name}, k={best_k}')
print(f'Silhouette: {assignment.metrics.silhouette:.3f}')
print(f'ARI: {assignment.metrics.ari_vs_reference:.3f}')
print(f'NMI: {assignment.metrics.nmi_vs_reference:.3f}')
print(f'Cohort entropy: {assignment.metrics.cohort_entropy_mean:.3f}')
print(f'Cramer\'s V: {assignment.metrics.cramers_v:.3f}')

Best: Composite, k=3
Silhouette: 0.211
ARI: 0.508
NMI: 0.538
Cohort entropy: 0.524
Cramer's V: 0.784


In [14]:
# Cluster x cohort contingency
from face_stratification.visualization.stage_b_plots import plot_cluster_cohort_contingency

fig = plot_cluster_cohort_contingency(
    assignment.labels.values, cohort_labels,
    output_path=OUTPUT_DIR / 'cluster_cohort_contingency.png',
)
fig.show()

choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
choreographer.utils._tmpfile | shutil.rmtree worked.
choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp9xqlxnva.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp7b_fhqz9.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp7b_fhqz9
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp9xqlxnva/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navigates done, putting them all in queue.
kaleido.kaleido | Getting tab from queue (has 1)
kaleido.kaleido | Got 45C5


In [15]:
# Information-theoretic analysis
from face_stratification.evaluation.validation import compute_information_theoretic_validation

info = compute_information_theoretic_validation(
    assignment.labels.values, cohort_labels,
    feature_matrix=train_Xn.to_numpy(),
    feature_names=list(train_Xn.columns),
)
print(f'Cluster entropy:       H(C) = {info["cluster_entropy"]:.3f} bits')
print(f'Cohort entropy:        H(R) = {info["cohort_entropy"]:.3f} bits')
print(f'H(cohort|cluster):     {info["h_cohort_given_cluster"]:.3f} bits')
print(f'H(cluster|cohort):     {info["h_cluster_given_cohort"]:.3f} bits')
print(f'Mutual information:    I = {info["mutual_information"]:.3f} bits')
print(f'Transdiagnostic score: {info["transdiagnostic_score"]:.3f}')

Cluster entropy:       H(C) = 1.518 bits
Cohort entropy:        H(R) = 1.592 bits
H(cohort|cluster):     0.756 bits
H(cluster|cohort):     0.682 bits
Mutual information:    I = 0.836 bits
Transdiagnostic score: 0.378


## 7. Cohort Fairness

In [16]:
from face_stratification.evaluation.fairness import cohort_fairness_metrics

fairness = cohort_fairness_metrics(assignment.labels.values, cohort_labels)
print(f'Entropy ratio (1.0 = perfectly transdiagnostic): {fairness["entropy_ratio"]:.3f}')
print(f'Max cluster imbalance ratio: {fairness["max_cluster_imbalance"]:.2f}x')

Entropy ratio (1.0 = perfectly transdiagnostic): 0.475
Max cluster imbalance ratio: 4.50x


## 8. DSM Subtype Comparison

In [17]:
from face_stratification.visualization.stage_b_plots import plot_dsm_subtype_mosaic

dsm_subtypes = train_ds.metadata['dsm_diagnosis'].values
fig = plot_dsm_subtype_mosaic(
    assignment.labels.values, dsm_subtypes,
    output_path=OUTPUT_DIR / 'dsm_subtype_alluvial.png',
)
fig.show()

choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpziuwxbhu.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp74j4qp7q.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp74j4qp7q
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpziuwxbhu/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navigates done, putting them all in queue.
kaleido.kaleido | Getting tab from queue (has 1)
kaleido.kaleido | Got 0B36
kaleido._kaleido_tab | Processing DSM_Subtypes__Data_Driven_Clusters.png
kaleido._kaleido_tab | Sending big command for D

### 8.1 UMAP projection of clusters

2D projection of the best embedding, colored two ways: (a) by **DSM cohort** (ground truth nosology),
(b) by **data-driven cluster**. If the two colorings match, the clustering is DSM-aligned
(not transdiagnostic). If they differ, the clustering is capturing something beyond nosology.

In [18]:
# Project the best embedding to 2D via UMAP (falls back to PCA if umap-learn unavailable)
try:
    import umap
    reducer = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                        metric='cosine', random_state=42)
    coords = reducer.fit_transform(best_emb.values.to_numpy())
    proj_name = 'UMAP'
except ImportError:
    from sklearn.decomposition import PCA
    coords = PCA(n_components=2, random_state=42).fit_transform(best_emb.values.to_numpy())
    proj_name = 'PCA (UMAP unavailable)'

print(f'{proj_name} projection: {coords.shape}')


choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
choreographer.utils._tmpfile | shutil.rmtree worked.


UMAP projection: (8811, 2)


In [19]:
# Side-by-side: coloring by cohort vs coloring by cluster
import plotly.graph_objects as go
from plotly.subplots import make_subplots

DARK = dict(template='plotly_dark', paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27')
COHORT_COLORS = {'bp': '#636EFA', 'sz': '#EF553B', 'dr': '#00CC96', 'asp': '#FFA15A'}
CLUSTER_COLORS = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA',
                  '#FF6692', '#B6E880', '#FF97FF', '#FECB52', '#19D3F3']

cluster_labels = assignment.labels.values

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f'{proj_name} colored by DSM cohort',
                    f'{proj_name} colored by data-driven cluster (k={best_k})'),
    horizontal_spacing=0.05,
)

# Left: DSM cohort
for cohort in sorted(set(cohort_labels)):
    mask = cohort_labels == cohort
    fig.add_trace(go.Scattergl(
        x=coords[mask, 0], y=coords[mask, 1],
        mode='markers',
        marker=dict(size=3, color=COHORT_COLORS.get(cohort, '#888'), opacity=0.55),
        name=cohort.upper(),
        legendgroup='cohort', legendgrouptitle_text='DSM cohort',
    ), row=1, col=1)

# Right: data-driven clusters
for c in sorted(set(cluster_labels)):
    mask = cluster_labels == c
    fig.add_trace(go.Scattergl(
        x=coords[mask, 0], y=coords[mask, 1],
        mode='markers',
        marker=dict(size=3, color=CLUSTER_COLORS[int(c) % len(CLUSTER_COLORS)], opacity=0.55),
        name=f'Cluster {c}',
        legendgroup='cluster', legendgrouptitle_text='Data-driven cluster',
    ), row=1, col=2)

fig.update_layout(**DARK, height=600, width=1400,
                  title_text=f'Patient embedding — DSM vs Data-Driven ({best_name}, k={best_k})')
fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.write_image(str(OUTPUT_DIR / 'umap_cohort_vs_cluster.png'), width=1400, height=600, scale=3)
fig.show()


choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp4zgs8912.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmppcei56e_.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmppcei56e_
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp4zgs8912/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navigates done, putting them all in queue.
kaleido.kaleido | Getting tab from queue (has 1)
kaleido.kaleido | Got 840C
kaleido

In [20]:
# For all fitted embeddings, project to 2D and show small multiples colored by cluster
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import umap
    def _project(X):
        return umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                         metric='cosine', random_state=42).fit_transform(X)
    proj_kind = 'UMAP'
except ImportError:
    from sklearn.decomposition import PCA
    def _project(X):
        return PCA(n_components=2, random_state=42).fit_transform(X)
    proj_kind = 'PCA'

n_methods = len(embeddings)
n_cols = min(3, n_methods)
n_rows = (n_methods + n_cols - 1) // n_cols

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=list(embeddings.keys()),
    horizontal_spacing=0.05, vertical_spacing=0.1,
)

for idx, (name, emb) in enumerate(embeddings.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    pts = _project(emb.values.to_numpy())
    # Use the best k for this method
    k = int(sweep_results[name].loc[sweep_results[name]['silhouette'].idxmax(), 'k'])
    from sklearn.cluster import KMeans
    labs = KMeans(n_clusters=k, random_state=0, n_init=10).fit_predict(emb.values.to_numpy())

    for c in sorted(set(labs)):
        mask = labs == c
        fig.add_trace(go.Scattergl(
            x=pts[mask, 0], y=pts[mask, 1],
            mode='markers',
            marker=dict(size=2.5, color=CLUSTER_COLORS[int(c) % len(CLUSTER_COLORS)], opacity=0.55),
            name=f'{name} c{c}',
            showlegend=False,
        ), row=row, col=col)

fig.update_layout(**DARK, height=350 * n_rows, width=400 * n_cols,
                  title_text=f'{proj_kind} projections per method (colored by cluster at best k)')
fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.write_image(str(OUTPUT_DIR / 'umap_all_methods.png'),
                width=400 * n_cols, height=350 * n_rows, scale=2)
fig.show()


choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
choreographer.utils._tmpfile | shutil.rmtree worked.
choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpef465x85.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmptpvkqqo_.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmptpvkqqo_
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpef465x85/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navig

## 8.2 Multipartite (bipartite/tripartite) embedding

The 4-way transdiagnostic subset has only **9 features** (demographics + substance use + comorbidity counts).
But the real signal is in features shared between 2 or 3 cohorts. Two ways to count these:

**Pattern-exact** (features unique to each cohort subset):
| Partition | # features |
|---|---|
| 4-way | 9 |
| bp+dr+sz | 20 |
| bp+dr | 17 |
| bp+sz | 5 |
| dr+sz | 4 |

**Cumulative** (features available in the partition's cohort subset, counting all that qualify):
| Partition | # features | Clinical meaning |
|---|---|---|
| bp+dr+sz (3-way) | **29** | Affective severity (YMRS, CGI-S, suicide items) + demographics |
| **bp+dr (2-way)** | **46** | MADRS, QIDS, STAI, Mathys + severity + demographics |
| **bp+sz (2-way)** | **37** | CVLT, fluency, WAIS + severity + demographics |
| **dr+sz (2-way)** | **33** | Glucose, lipids, TG/HDL + severity + demographics |
| asp+bp+sz (3-way) | 12 | EQ-5D + treatment flags + demographics |
| asp+dr (2-way) | 10 | Mostly demographics + EGF |

**Cumulative is the correct mode for bipartite/tripartite graph construction**: a BP+DR graph should use
every feature available in both BP *and* DR patients (regardless of whether those features also happen to
be present in SZ or ASP). Pattern-exact is useful only for reporting unique contributions.

The module supports both modes via the `feature_mode` parameter; `cumulative` is the default.

In [21]:
from face_stratification.graph.multipartite import (
    MultipartiteSpectralEmbedding, identify_coverage_partitions,
)

# Identify all coverage partitions at 30% threshold
partitions = identify_coverage_partitions(
    dataset,
    min_coverage_in_subset=0.30,
    min_features_per_partition=3,
)

rows = []
for p in partitions:
    rows.append({
        'Partition': p.name,
        '# cohorts': len(p.cohorts),
        '# features': p.n_features,
        'Example features': ', '.join(p.features[:3]) + ('...' if p.n_features > 3 else ''),
    })
pd.DataFrame(rows)

face_stratification.graph.multipartite | Identified 11 partitions (mode=cumulative, coverage>=30%, min_features=3)
face_stratification.graph.multipartite |   ASP+BP+DR+SZ (4 cohorts, 9 features)
face_stratification.graph.multipartite |   BP+DR+SZ (3 cohorts, 29 features)
face_stratification.graph.multipartite |   ASP+BP+SZ (3 cohorts, 12 features)
face_stratification.graph.multipartite |   ASP+BP+DR (3 cohorts, 9 features)
face_stratification.graph.multipartite |   ASP+DR+SZ (3 cohorts, 9 features)
face_stratification.graph.multipartite |   BP+DR (2 cohorts, 46 features)
face_stratification.graph.multipartite |   BP+SZ (2 cohorts, 37 features)
face_stratification.graph.multipartite |   DR+SZ (2 cohorts, 33 features)
face_stratification.graph.multipartite |   ASP+BP (2 cohorts, 12 features)
face_stratification.graph.multipartite |   ASP+SZ (2 cohorts, 12 features)
face_stratification.graph.multipartite |   ASP+DR (2 cohorts, 10 features)


,Partition,# cohorts,# features,Example features
0,asp+bp+dr+sz,4,9,"cm_n_psychiatric, cm_n_somatic, demo_age_years..."
1,bp+dr+sz,3,29,"bio_bmi, cm_n_psychiatric, cm_n_somatic..."
2,asp+bp+sz,3,12,"cm_n_psychiatric, cm_n_somatic, demo_age_years..."
3,asp+bp+dr,3,9,"cm_n_psychiatric, cm_n_somatic, demo_age_years..."
4,asp+dr+sz,3,9,"cm_n_psychiatric, cm_n_somatic, demo_age_years..."
5,bp+dr,2,46,"bio_bmi, bio_dbp_mmhg, bio_sbp_mmhg..."
6,bp+sz,2,37,"bio_bmi, cm_n_psychiatric, cm_n_somatic..."
7,dr+sz,2,33,"bio_bmi, bio_fasting_glucose, bio_hdl_choleste..."
8,asp+bp,2,12,"cm_n_psychiatric, cm_n_somatic, demo_age_years..."
9,asp+sz,2,12,"cm_n_psychiatric, cm_n_somatic, demo_age_years..."


In [22]:
# Fit multipartite embedding on the TRAIN split (using the already-normalized train_ds_norm)
import time

mp_model = MultipartiteSpectralEmbedding(
    min_coverage=0.30,
    min_features_per_partition=3,
    n_components_per_partition=8,
    k_neighbours=10,
    include_4way=True,
    include_mask_columns=True,
    l2_normalize=True,
)

t0 = time.time()
mp_emb = mp_model.fit_transform(train_ds_norm)
print(f'Multipartite: {mp_emb.n_patients} patients × {mp_emb.dim} dims in {time.time()-t0:.1f}s')
print(f'\nView dims per partition:')
for view, dim in mp_emb.view_dims.items():
    print(f'  {view:30s}  {dim} dims')

face_stratification.graph.multipartite | Identified 11 partitions (mode=cumulative, coverage>=30%, min_features=3)
face_stratification.graph.multipartite |   ASP+BP+DR+SZ (4 cohorts, 9 features)
face_stratification.graph.multipartite |   BP+DR+SZ (3 cohorts, 29 features)
face_stratification.graph.multipartite |   ASP+BP+SZ (3 cohorts, 11 features)
face_stratification.graph.multipartite |   ASP+BP+DR (3 cohorts, 9 features)
face_stratification.graph.multipartite |   ASP+DR+SZ (3 cohorts, 9 features)
face_stratification.graph.multipartite |   BP+DR (2 cohorts, 46 features)
face_stratification.graph.multipartite |   BP+SZ (2 cohorts, 37 features)
face_stratification.graph.multipartite |   DR+SZ (2 cohorts, 33 features)
face_stratification.graph.multipartite |   ASP+BP (2 cohorts, 11 features)
face_stratification.graph.multipartite |   ASP+SZ (2 cohorts, 11 features)
face_stratification.graph.multipartite |   ASP+DR (2 cohorts, 10 features)
face_stratification.graph.multipartite | Partitio

Multipartite: 8811 patients × 99 dims in 61.4s

View dims per partition:
  asp+bp+dr+sz                    8 dims
  asp+bp+dr+sz_mask               1 dims
  bp+dr+sz                        8 dims
  bp+dr+sz_mask                   1 dims
  asp+bp+sz                       8 dims
  asp+bp+sz_mask                  1 dims
  asp+bp+dr                       8 dims
  asp+bp+dr_mask                  1 dims
  asp+dr+sz                       8 dims
  asp+dr+sz_mask                  1 dims
  bp+dr                           8 dims
  bp+dr_mask                      1 dims
  bp+sz                           8 dims
  bp+sz_mask                      1 dims
  dr+sz                           8 dims
  dr+sz_mask                      1 dims
  asp+bp                          8 dims
  asp+bp_mask                     1 dims
  asp+sz                          8 dims
  asp+sz_mask                     1 dims
  asp+dr                          8 dims
  asp+dr_mask                     1 dims


In [23]:
# Run multi-metric sweep on the multipartite embedding
mp_sweep = multi_metric_sweep(mp_emb.values, cohort_labels, k_values=range(3, 13))

# Add it alongside existing methods
multi_metric_results['Multipartite'] = mp_sweep
sweep_results['Multipartite'] = mp_sweep.rename(columns={'davies_bouldin': 'db'})
embeddings['Multipartite'] = mp_emb

# Best k per metric
print(f"Multipartite — best k by metric:")
print(f"  silhouette:  k={int(mp_sweep.loc[mp_sweep['silhouette'].idxmax(), 'k'])}  (sil={mp_sweep['silhouette'].max():.3f})")
print(f"  DB (lower):  k={int(mp_sweep.loc[mp_sweep['davies_bouldin'].idxmin(), 'k'])}  (DB={mp_sweep['davies_bouldin'].min():.3f})")
print(f"  composite:   k={int(mp_sweep.loc[mp_sweep['composite'].idxmax(), 'k'])}  (comp={mp_sweep['composite'].max():.3f})")
print(f"\nFull sweep:")
mp_sweep[['k', 'silhouette', 'davies_bouldin', 'cramers_v', 'td_ratio', 'composite']].round(3)

Multipartite — best k by metric:
  silhouette:  k=8  (sil=0.324)
  DB (lower):  k=3  (DB=1.848)
  composite:   k=6  (comp=0.273)

Full sweep:


,k,silhouette,davies_bouldin,cramers_v,td_ratio,composite
0,3,0.322,1.848,1.000,0.068,0.242
1,4,0.263,2.224,0.827,0.072,0.232
2,5,0.282,2.185,0.853,0.093,0.263
3,6,0.289,2.007,0.869,0.086,0.273
4,7,0.321,1.855,0.995,0.025,0.198
5,8,0.324,1.855,0.991,0.035,0.214
6,9,0.299,1.953,0.992,0.030,0.172
7,10,0.297,1.913,0.990,0.035,0.180
8,11,0.291,1.907,0.993,0.023,0.161
9,12,0.283,1.873,0.993,0.022,0.156


In [24]:
# Pick best k by the composite score (not silhouette alone)
mp_best_k = int(mp_sweep.loc[mp_sweep['composite'].idxmax(), 'k'])
mp_assignment = run_kmeans(mp_emb.values, n_clusters=mp_best_k, reference_labels=cohort_labels)

print(f'Multipartite clustering (k={mp_best_k}, composite-selected):')
print(f'  Silhouette:     {mp_assignment.metrics.silhouette:.3f}')
print(f'  Davies-Bouldin: {mp_assignment.metrics.davies_bouldin:.3f}')
print(f'  ARI vs DSM:     {mp_assignment.metrics.ari_vs_reference:.3f}  (lower = more transdiagnostic)')
print(f'  NMI vs DSM:     {mp_assignment.metrics.nmi_vs_reference:.3f}')
print(f"  Cramer V:       {mp_assignment.metrics.cramers_v:.3f}  (lower = more transdiagnostic)")
print(f'  Cohort entropy: {mp_assignment.metrics.cohort_entropy_mean:.3f}  (higher = more transdiagnostic)')

Multipartite clustering (k=6, composite-selected):
  Silhouette:     0.289
  Davies-Bouldin: 2.007
  ARI vs DSM:     0.422  (lower = more transdiagnostic)
  NMI vs DSM:     0.690
  Cramer V:       0.869  (lower = more transdiagnostic)
  Cohort entropy: 0.119  (higher = more transdiagnostic)


In [25]:
# UMAP visualization: cohort vs multipartite cluster
try:
    import umap
    mp_coords = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.3,
                          metric='cosine', random_state=42).fit_transform(mp_emb.values.to_numpy())
    proj_name = 'UMAP'
except ImportError:
    from sklearn.decomposition import PCA
    mp_coords = PCA(n_components=2, random_state=42).fit_transform(mp_emb.values.to_numpy())
    proj_name = 'PCA'

import plotly.graph_objects as go
from plotly.subplots import make_subplots

DARK = dict(template='plotly_dark', paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27')
COHORT_COLORS = {'bp': '#636EFA', 'sz': '#EF553B', 'dr': '#00CC96', 'asp': '#FFA15A'}
CLUSTER_COLORS = ['#636EFA', '#EF553B', '#00CC96', '#FFA15A', '#AB63FA',
                  '#FF6692', '#B6E880', '#FF97FF', '#FECB52', '#19D3F3']

mp_labels = mp_assignment.labels.values

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(f'Multipartite {proj_name} colored by DSM cohort',
                    f'Multipartite {proj_name} colored by data-driven cluster (k={mp_best_k})'),
    horizontal_spacing=0.05,
)

for cohort in sorted(set(cohort_labels)):
    mask = cohort_labels == cohort
    fig.add_trace(go.Scattergl(
        x=mp_coords[mask, 0], y=mp_coords[mask, 1], mode='markers',
        marker=dict(size=3, color=COHORT_COLORS.get(cohort, '#888'), opacity=0.55),
        name=cohort.upper(), legendgroup='cohort', legendgrouptitle_text='DSM cohort',
    ), row=1, col=1)

for c in sorted(set(mp_labels)):
    mask = mp_labels == c
    fig.add_trace(go.Scattergl(
        x=mp_coords[mask, 0], y=mp_coords[mask, 1], mode='markers',
        marker=dict(size=3, color=CLUSTER_COLORS[int(c) % len(CLUSTER_COLORS)], opacity=0.55),
        name=f'Cluster {c}', legendgroup='cluster', legendgrouptitle_text='Multipartite cluster',
    ), row=1, col=2)

fig.update_layout(**DARK, height=600, width=1400,
                  title_text='Multipartite embedding — DSM vs Data-Driven clusters')
fig.update_xaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.update_yaxes(showticklabels=False, showgrid=False, zeroline=False)
fig.write_image(str(OUTPUT_DIR / 'multipartite_umap.png'), width=1400, height=600, scale=3)
fig.show()

choreographer.browsers.chromium | Chromium init'ed with kwargs {}
choreographer.browsers.chromium | Found chromium path: /Applications/Google Chrome.app/Contents/MacOS/Google Chrome
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpyd_4x_eg.
choreographer.browser_async | Opening browser.
choreographer.utils._tmpfile | Temp directory created: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp0k9tx_wi.
choreographer.browsers.chromium | Temporary directory at: /var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmp0k9tx_wi
Numba: Attempted to fork from a non-main thread, the TBB library may be in an invalid state in the child process.
kaleido.kaleido | Conforming 1 to file:///var/folders/hh/5fv05dss32j5d807znp33_fc0000gn/T/tmpyd_4x_eg/index.html
kaleido.kaleido | Waiting on all navigates
kaleido.kaleido | All navigates done, putting them all in queue.
kaleido.kaleido | Getting tab from queue (has 1)
kaleido.kaleido | Got 5D4B
kaleido

In [26]:
# Per-partition contribution: which partitions drive the clustering?
# For each partition, compute ANOVA F-statistic of its columns against cluster labels
from scipy.stats import f_oneway

rows = []
for partition_name, dim in mp_emb.view_dims.items():
    cols = [c for c in mp_emb.values.columns if c.startswith(f'{partition_name}::')]
    if not cols:
        continue
    f_stats = []
    for col in cols:
        vals = mp_emb.values[col].to_numpy()
        groups = [vals[mp_labels == c] for c in np.unique(mp_labels)]
        groups = [g for g in groups if len(g) > 2 and np.std(g) > 0]
        if len(groups) >= 2:
            try:
                f, _ = f_oneway(*groups)
                if np.isfinite(f):
                    f_stats.append(f)
            except Exception:
                pass
    rows.append({
        'Partition': partition_name,
        'N dims': dim,
        'Mean F-stat': np.mean(f_stats) if f_stats else np.nan,
        'Max F-stat': np.max(f_stats) if f_stats else np.nan,
    })

contrib = pd.DataFrame(rows).sort_values('Mean F-stat', ascending=False)
print('Per-partition contribution to cluster separation (higher F = more driving):')
contrib.round(1)

choreographer.utils._tmpfile | TemporaryDirectory.cleanup() worked.
choreographer.utils._tmpfile | shutil.rmtree worked.


Per-partition contribution to cluster separation (higher F = more driving):


,Partition,N dims,Mean F-stat,Max F-stat
9,asp+sz,8,35246.5,315677.8
0,asp+bp+dr+sz,8,6819.1,61156.2
3,asp+bp+dr,8,3265.5,27922.5
4,asp+dr+sz,8,2012.0,15665.0
10,asp+dr,8,1509.8,11374.7
1,bp+dr+sz,8,1317.0,4609.3
7,dr+sz,8,919.0,7090.4
6,bp+sz,8,918.7,1897.8
8,asp+bp,8,749.3,4194.3
2,asp+bp+sz,8,688.0,4277.4


In [27]:
# Cohort composition per cluster — is any cluster genuinely multi-cohort?
ct = pd.crosstab(
    pd.Series(mp_labels, name='Cluster'),
    pd.Series(cohort_labels, name='Cohort'),
    normalize='index',
)
print(f'Multipartite clusters — cohort composition (row-normalized):')
display(ct.round(3))

# Identify "genuinely transdiagnostic" clusters (no cohort > 60%)
transdiag_clusters = ct.index[(ct.max(axis=1) < 0.6)].tolist()
print(f'\nGenuinely transdiagnostic clusters (no cohort > 60%): {transdiag_clusters}')
print(f'Cohort-dominated clusters: {[c for c in ct.index if c not in transdiag_clusters]}')

Multipartite clusters — cohort composition (row-normalized):


Cohort,asp,bp,dr,sz
Cluster,,,,
0,0.0,0.656,0.337,0.007
1,0.0,0.999,0.001,0.001
2,0.0,0.000,0.000,1.000
3,0.0,0.999,0.000,0.001
4,0.0,0.997,0.000,0.003
5,1.0,0.000,0.000,0.000



Genuinely transdiagnostic clusters (no cohort > 60%): []
Cohort-dominated clusters: [0, 1, 2, 3, 4, 5]


## 9. Summary Table

In [28]:
# Build summary across all methods
rows = []
for name, emb in embeddings.items():
    sweep = sweep_results[name]
    best_k = int(sweep.loc[sweep['silhouette'].idxmax(), 'k'])
    a = run_kmeans(emb.values, n_clusters=best_k, reference_labels=cohort_labels)
    m = a.metrics
    rows.append({
        'Method': name,
        'Dim': emb.dim,
        'Best k': best_k,
        'Silhouette': m.silhouette,
        'ARI': m.ari_vs_reference,
        'NMI': m.nmi_vs_reference,
        'Cramer V': m.cramers_v,
        'Entropy': m.cohort_entropy_mean,
        'CH': m.calinski_harabasz,
        'DB': m.davies_bouldin,
    })

summary = pd.DataFrame(rows).set_index('Method')
summary.style.format('{:.3f}', subset=[
    'Silhouette', 'ARI', 'NMI', 'Cramer V', 'Entropy', 'DB'
]).format('{:.0f}', subset=['CH'])

,Dim,Best k,Silhouette,ARI,NMI,Cramer V,Entropy,CH,DB
Method,,,,,,,,,
PCA,8,6,0.423,0.055,0.169,0.402,0.815,2653,1.282
KernelPCA,16,9,0.441,0.140,0.298,0.571,0.485,1912,1.216
Spectral (TD),16,12,0.156,0.015,0.025,0.167,1.058,397,2.223
Spectral (Multiplex),32,3,0.489,0.743,0.673,0.879,0.460,2950,1.825
Composite,56,3,0.211,0.508,0.538,0.784,0.524,1013,2.638
Multipartite,99,8,0.324,0.394,0.686,0.991,0.049,985,1.855


## 10. Full Pipeline (optional)

Run the complete 16-method pipeline via `StageBPipeline`.

In [ ]:
# Uncomment to run the full pipeline (takes ~30-60 minutes with GNNs)
#
# from face_stratification.stage_b.pipeline import StageBPipeline
#
# pipeline = StageBPipeline(output_dir='../output/stage_b')
# result = pipeline.run(dataset, skip_gnn=False, skip_permutation=False)
# full_summary = pipeline.summarize(result)
# display(full_summary)